In [ ]:
"""
Prediction Script - matches train_model.py preprocessing exactly.
Reuses the saved encoders, scaler, and locked column order so nothing
gets silently misaligned before it hits the model.
"""

import pickle
import numpy as np
import pandas as pd
import tensorflow as tf


# ---------------------------------------------------------------------------
# 1. LOAD MODEL + SAVED PREPROCESSING ARTIFACTS
# ---------------------------------------------------------------------------
model = tf.keras.models.load_model('model.keras')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

# ---------------------------------------------------------------------------
# 2. SAMPLE INPUT
# ---------------------------------------------------------------------------
input_data = {
    'CreditScore': 400,
    'Geography': 'Germany',
    'Gender': 'Male',
    'Age': 4035,
    'Tenure': 3,
    'Balance': 10000,
    'NumOfProducts': 3,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 75000
}

input_df = pd.DataFrame([input_data])

# ---------------------------------------------------------------------------
# 3. ENCODE  (identical steps and identical fitted encoders as training)
# ---------------------------------------------------------------------------
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)

# ---------------------------------------------------------------------------
# 4. REORDER COLUMNS TO MATCH TRAINING EXACTLY
#    (this is the step the original code skipped -- without it, a scaler
#    fit on one column order silently scales the wrong values)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 5. SCALE + PREDICT
# ---------------------------------------------------------------------------
input_scaled = scaler.transform(input_df)
prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]

print(f"Churn probability: {prediction_proba:.4f}")
print("Prediction:", "Customer will churn" if prediction_proba > 0.5 else "Customer will stay")


1/1 [==============================] - 0s 34ms/step
Churn probability: 1.0000
Prediction: Customer will churn
